# Wildlife Species Classification

PyTorch + timm + Albumentations

## 2. Project Overview

Build a wildlife species classifier for an iNaturalist-style or wildlife-camera-trap subset, with explicit class imbalance handling and macro-F1 as the main evaluation metric.

## 3. Learning Objectives

- Train a transfer-learning wildlife classifier
- Handle long-tail species imbalance safely
- Evaluate with macro-F1 and class-wise recall
- Understand why accuracy can be misleading on wildlife datasets

## 4. Problem Statement

Given an animal image, predict the species class from a multi-class wildlife label set.

## 5. Why This Project Matters

Wildlife classification supports biodiversity monitoring, camera-trap analytics, and ecological research where rare-species performance matters more than headline accuracy.

## 6. Dataset Overview

Target dataset: an iNaturalist subset or wildlife dataset with multiple species classes and typically strong long-tail imbalance.

## 7. Dataset Source and License Notes

Follow the original dataset license, attribution rules, and any use restrictions for images gathered from citizen science or camera traps.

## 8. Environment Setup

Required: torch, timm, albumentations, scikit-learn, matplotlib, seaborn.

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('timm:', timm.__version__)
print('Albumentations:', A.__version__)

In [ ]:
# 10. Configuration / constants
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
LR = 1e-4
EPOCHS_BASELINE = 1
EPOCHS_STRONG = 1

BASELINE_MODEL = 'resnet18'
STRONG_MODEL = 'convnext_tiny'

SAVE_DIR = Path.cwd() / 'Computer Vision' / 'Wildlife Species Classification'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = SAVE_DIR / 'wildlife_species_data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
print('Save dir:', SAVE_DIR)

In [ ]:
# 11. Dataset loading
FORCE_SYNTHETIC = True
use_synthetic = FORCE_SYNTHETIC

# Example wildlife classes; replace with actual dataset labels when using real data.
class_names = ['elephant', 'lion', 'zebra', 'giraffe', 'leopard', 'buffalo', 'rhino', 'hyena']
num_classes = len(class_names)

train_images, train_labels = [], []
val_images, val_labels = [], []
test_images, test_labels = [], []

if not use_synthetic:
    # Expected real-data layout example:
    # DATA_DIR/train/<class_name>/*.jpg
    # DATA_DIR/val/<class_name>/*.jpg
    # DATA_DIR/test/<class_name>/*.jpg
    pass

if use_synthetic:
    # Long-tail style imbalance to mirror wildlife datasets
    train_counts = [900, 420, 760, 510, 180, 690, 95, 130]
    val_counts = [130, 60, 110, 75, 30, 95, 15, 20]
    test_counts = [180, 85, 145, 100, 40, 130, 20, 28]

    for cls, n in enumerate(train_counts):
        for _ in range(n):
            train_labels.append(cls)
            train_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

    for cls, n in enumerate(val_counts):
        for _ in range(n):
            val_labels.append(cls)
            val_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

    for cls, n in enumerate(test_counts):
        for _ in range(n):
            test_labels.append(cls)
            test_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

print('Using synthetic mode:', use_synthetic)
print('Classes:', class_names)
print('Train/Val/Test sizes:', len(train_labels), len(val_labels), len(test_labels))

In [ ]:
# 12. Data validation checks
assert len(train_images) == len(train_labels)
assert len(val_images) == len(val_labels)
assert len(test_images) == len(test_labels)
assert set(train_labels).issubset(set(range(num_classes)))

print('Validation checks passed.')
print('Train class counts:', dict(zip(class_names, np.bincount(np.array(train_labels), minlength=num_classes).tolist())))

## 13. Exploratory Data Analysis

Inspect the long-tail class distribution and sample wildlife images.

In [ ]:
counts = np.bincount(np.array(train_labels), minlength=num_classes)

plt.figure(figsize=(10, 4))
plt.bar(class_names, counts)
plt.xticks(rotation=30, ha='right')
plt.title('Train Class Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print('Imbalance ratio max/min:', round(float((counts.max()+1)/(counts.min()+1)), 2))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    idx = np.where(np.array(train_labels) == i)[0][0]
    ax.imshow(train_images[idx])
    ax.set_title(class_names[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 14. Train/Validation/Test Split Strategy

Use stratified splits and preserve rare-species representation. If the dataset has camera or sequence metadata, avoid placing adjacent frames from the same event in multiple splits.

In [ ]:
# 15. Preprocessing and augmentation strategy
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.10, rotate_limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.4),
    A.HueSaturationValue(10, 15, 10, p=0.25),
    A.CoarseDropout(max_holes=6, max_height=24, max_width=24, p=0.2),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

print('Albumentations pipelines configured.')

## 16. Baseline Approach

Baseline transfer-learning model: ResNet-18.

In [ ]:
class WildlifeDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        y = int(self.labels[idx])
        x = self.transform(image=img)['image']
        return x, y

train_ds = WildlifeDataset(train_images, train_labels, train_tf)
val_ds = WildlifeDataset(val_images, val_labels, val_tf)
test_ds = WildlifeDataset(test_images, test_labels, val_tf)

# Imbalance handling: weighted sampling
class_counts = np.bincount(np.array(train_labels), minlength=num_classes)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = np.array([class_weights[y] for y in train_labels], dtype=np.float32)
sampler = WeightedRandomSampler(weights=torch.from_numpy(sample_weights), num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

baseline_model = timm.create_model(BASELINE_MODEL, pretrained=True, num_classes=num_classes).to(DEVICE)
strong_model = timm.create_model(STRONG_MODEL, pretrained=True, num_classes=num_classes).to(DEVICE)

print('Baseline model:', BASELINE_MODEL)
print('Strong model:', STRONG_MODEL)
print('Weighted sampler enabled for training.')

## 17. Main Model/Workflow

Stronger model: ConvNeXt-Tiny transfer learning.

In [ ]:
# 18. Training loop / fine-tuning pipeline
def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    counts = np.bincount(np.array(train_labels), minlength=num_classes)
    weights = torch.tensor(1.0 / np.maximum(counts, 1), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)

    total_loss = 0.0
    ys, ps = [], []

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train_mode:
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item())
        pred = logits.argmax(dim=1)
        ys.extend(yb.cpu().numpy().tolist())
        ps.extend(pred.cpu().numpy().tolist())

    avg_loss = total_loss / max(len(loader), 1)
    acc = accuracy_score(ys, ps)
    macro_f1 = f1_score(ys, ps, average='macro', zero_division=0)
    macro_recall = recall_score(ys, ps, average='macro', zero_division=0)
    return avg_loss, acc, macro_f1, macro_recall, ys, ps


def train_model(model, name, epochs):
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    best_f1 = -1.0
    best_state = None

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc, tr_f1, tr_rec, _, _ = run_epoch(model, train_loader, optimizer=optimizer)
        va_loss, va_acc, va_f1, va_rec, _, _ = run_epoch(model, val_loader, optimizer=None)
        print(f'[{name}] ep={ep} train_acc={tr_acc:.4f} train_macro_f1={tr_f1:.4f} val_acc={va_acc:.4f} val_macro_f1={va_f1:.4f}')
        if va_f1 > best_f1:
            best_f1 = va_f1
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

train_model(baseline_model, 'baseline', EPOCHS_BASELINE)
train_model(strong_model, 'strong', EPOCHS_STRONG)

## 19. Inference Examples

Deployment-style top-k prediction response for one wildlife image.

In [ ]:
def infer_single(model, image_rgb, k=3):
    model.eval()
    x = val_tf(image=image_rgb)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    top_idx = np.argsort(-probs)[:k]
    preds = []
    for i in top_idx:
        preds.append({
            'class_id': int(i),
            'class_name': class_names[int(i)],
            'confidence': float(probs[int(i)])
        })

    return {
        'top_k': preds,
        'predicted_class': preds[0]['class_name'],
        'confidence': preds[0]['confidence']
    }

sample = test_images[0]
resp = infer_single(strong_model, sample, k=3)
print(json.dumps(resp, indent=2))

## 20. Evaluation with Macro-F1 Emphasis

Macro-F1 is the main metric because rare species matter and accuracy can be dominated by common classes.

In [ ]:
_, b_acc, b_f1, b_rec, by, bp = run_epoch(baseline_model, test_loader, optimizer=None)
_, s_acc, s_f1, s_rec, sy, sp = run_epoch(strong_model, test_loader, optimizer=None)

print('Baseline -> acc:', round(b_acc,4), 'macro_f1:', round(b_f1,4), 'macro_recall:', round(b_rec,4))
print('Strong   -> acc:', round(s_acc,4), 'macro_f1:', round(s_f1,4), 'macro_recall:', round(s_rec,4))

cls_recall = recall_score(sy, sp, average=None, labels=list(range(num_classes)), zero_division=0)
print('Class-wise recall:')
for i, r in enumerate(cls_recall):
    print(class_names[i], round(float(r), 4))

print()
print('Classification report (strong):')
print(classification_report(sy, sp, target_names=class_names, zero_division=0))

## 21. Error Analysis

Confusion matrix to inspect common cross-species confusions.

In [ ]:
cm = confusion_matrix(sy, sp, labels=list(range(num_classes)))
cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(8, 6))
sns.heatmap(cmn, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.title('Normalized Confusion Matrix (Strong Model)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

## 22. Class Imbalance Handling Summary

This notebook handles imbalance using:

1. **WeightedRandomSampler** to give minority species more training exposure
2. **Class-weighted cross-entropy** to penalize rare-species mistakes more strongly
3. **Macro-F1 emphasis** so each species contributes equally to the headline score

## 23. Why Macro-F1 Matters Here

Wildlife datasets are often long-tailed: a few common species dominate image counts while rare species have very little data. Accuracy can look strong even if the model ignores rare animals. Macro-F1 forces equal weight across classes and better reflects conservation usefulness.

## 24. How To Improve

- Add stronger long-tail methods such as focal loss or class-balanced loss
- Use larger backbones and longer training
- Incorporate metadata such as location/time when appropriate
- Add open-set rejection for unseen species

## 25. Production Considerations

- Monitor rare-species recall separately
- Audit performance by camera, geography, and lighting conditions
- Use human review for rare or high-importance species predictions

## 26. Common Mistakes

- Reporting only accuracy on long-tail wildlife data
- Ignoring correlated camera-burst leakage across splits
- Treating low-support species as negligible

## 27. Mini Challenge / Final Summary

Mini challenge: compare weighted sampling against plain random sampling and measure macro-F1 drop.

Summary: this notebook builds a wildlife species classifier with explicit class imbalance handling and macro-F1-first evaluation.

In [ ]:
# Save metrics
train_counts = np.bincount(np.array(train_labels), minlength=num_classes)

metrics = {
    'dataset': 'iNaturalist subset / Wildlife dataset',
    'num_classes': int(num_classes),
    'baseline_model': BASELINE_MODEL,
    'strong_model': STRONG_MODEL,
    'baseline_test_accuracy': float(b_acc),
    'baseline_test_macro_f1': float(b_f1),
    'baseline_test_macro_recall': float(b_rec),
    'strong_test_accuracy': float(s_acc),
    'strong_test_macro_f1': float(s_f1),
    'strong_test_macro_recall': float(s_rec),
    'class_wise_recall_strong': {class_names[i]: float(cls_recall[i]) for i in range(num_classes)},
    'imbalance_ratio_train_max_min': float((train_counts.max()+1)/(train_counts.min()+1)),
    'device': str(DEVICE),
    'synthetic_mode': bool(use_synthetic)
}

metrics_path = SAVE_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved metrics:', metrics_path)
print('Done.')